# Day 45: Cleaning, EDA and Visualisation

**Today's question:** can we turn the messy `students_raw.csv` into the clean `students.csv` with a few lines of
pandas, and then **show** what the data says with charts?

Plan: find the problems → clean in 4 steps → check against the official clean file → explore → 3 charts.
Before running a cell marked 🤔, predict what it will show.

## 1. Find the problems with pandas

On Day 42 the audit took dozens of lines. Here it is again in pandas. Blank cells are loaded as `NaN`
("not a number"), pandas' marker for a **missing** value.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../../../datasets")
raw = pd.read_csv(DATA_DIR / "students_raw.csv")
print(raw.shape)
raw.isna().sum()          # isna(): True where a value is missing; sum() counts per column

In [ ]:
print("duplicate rows:", raw.duplicated().sum())
raw[(raw["attendance_pct"] > 100) | (raw["study_hours"] < 0)]

🤔 **Predict:** why does `value_counts` below show **two** lines that both look like `arts`?

In [ ]:
raw["program"].value_counts()

One of them is `"arts "` with a trailing space. pandas does not show the quotes, so it looks identical.

## 2. The cleaning recipe

We apply exactly these four steps, in this order:

1. remove exact duplicate rows;
2. clean the text categories: strip spaces, lower case;
3. keep only rows with `attendance_pct <= 100` and a `study_hours` that is not negative;
4. fill missing `study_hours` and `sleep_hours` with the column **median**.

### Step 1: duplicates

In [ ]:
df = raw.drop_duplicates()
print(raw.shape, "→", df.shape)

### Step 2: inconsistent categories

`.str` (Day 44) applies a string method to every value. Methods can be chained: strip first, then lower.

In [ ]:
df["program"] = df["program"].str.strip().str.lower()
df["has_job"] = df["has_job"].str.strip().str.lower()
print(df["program"].value_counts())
print(df["has_job"].value_counts())

### Step 3: impossible values

🤔 **Predict:** after this filter, how many rows are left? (We expect to remove just 2 bad rows from 300.)

In [ ]:
attempt = df[(df["attendance_pct"] <= 100) & (df["study_hours"] >= 0)]
print(attempt.shape)

290, not 298! A missing value compared with anything gives `False`, so `study_hours >= 0` silently threw away
the 8 students whose study hours are **missing** (we want to fill those, not delete them).

Say exactly what we mean: *keep the row if study hours are not negative **or** missing.*

In [ ]:
keep = (df["attendance_pct"] <= 100) & ((df["study_hours"] >= 0) | df["study_hours"].isna())
df = df[keep]
print(df.shape)

### Step 4: missing values, drop or fill?

Option A, `dropna()`: remove every row that has any missing value.

In [ ]:
print("dropna would leave:", df.dropna().shape)

That deletes 14 students (5 % of the data) because one number each is missing. Option B, `fillna()`: replace
the gap with a typical value. We use the **median** because, unlike the mean, it is not pulled by extreme values (Day 43).

In [ ]:
for column in ["study_hours", "sleep_hours"]:
    median = df[column].median()
    df[column] = df[column].fillna(median)
    print(column, "filled with", median)

print("missing values left:", df.isna().sum().sum())

Filling is a **choice** with a cost: 8 students now have an invented study time of exactly 8.9 hours.
Always write down which choice you made and why.

## 3. Check: does it match `students.csv`?

In [ ]:
official = pd.read_csv(DATA_DIR / "students.csv")
print("mine:", df.shape, "| official:", official.shape)

comparison = pd.DataFrame({
    "mine": df.mean(numeric_only=True),           # numeric_only: skip the text columns
    "official": official.mean(numeric_only=True),
})
comparison.round(3)

In [ ]:
print("All column means identical:", all(comparison["mine"] == comparison["official"]))

Same shape, same means: our recipe reproduces the official clean file.

The whole recipe as one function, so it can be reused (e.g. in Project 06):

In [ ]:
def clean_students(raw):
    """Apply the 4-step cleaning recipe to the raw students table and return a new, clean table."""
    df = raw.drop_duplicates()
    for column in ["program", "has_job"]:
        df[column] = df[column].str.strip().str.lower()
    keep = (df["attendance_pct"] <= 100) & ((df["study_hours"] >= 0) | df["study_hours"].isna())
    df = df[keep]
    for column in ["study_hours", "sleep_hours"]:
        df[column] = df[column].fillna(df[column].median())
    return df


students = clean_students(raw)
print(students.shape, raw.shape)   # the raw table is unchanged

## 4. Explore before you plot (EDA)

**EDA** = exploratory data analysis: look at the numbers first, so you know what to plot.

In [ ]:
students[["study_hours", "attendance_pct", "sleep_hours", "final_score"]].describe().round(1)

In [ ]:
students.groupby("program")["final_score"].mean().round(1)

## 5. Charts with matplotlib

By convention: `import matplotlib.pyplot as plt`. Each chart: draw → title → axis labels → `plt.show()`.

### Histogram: how are values spread out?

In [ ]:
import matplotlib.pyplot as plt

plt.hist(students["final_score"], bins=20)
plt.title("Distribution of final scores")
plt.xlabel("Final score (0–100)")
plt.ylabel("Number of students")
plt.show()

🤔 **Discuss:** where are most students? Roughly what share is below the pass mark of 50?

### Bar chart: compare a number across groups

`groupby` gives a Series: its `.index` holds the group names (bar labels) and its `.values` the numbers (bar heights).

In [ ]:
pass_rate = students.groupby("has_job")["passed"].mean() * 100
print(pass_rate)

plt.bar(pass_rate.index, pass_rate.values)
plt.title("Pass rate by part-time job")
plt.xlabel("Has a part-time job")
plt.ylabel("Students who passed (%)")
plt.show()

### Scatter plot: do two numbers move together?

In [ ]:
plt.scatter(students["study_hours"], students["final_score"], alpha=0.5)   # alpha: see-through dots
plt.title("Study hours vs final score")
plt.xlabel("Study hours per week")
plt.ylabel("Final score")
plt.show()

🤔 **Discuss:** the dots drift upwards: more study, higher scores *on average*. Does this **prove** that studying more
*causes* higher scores? What else could explain it?

**Correlation ≠ causation.** Students who study more may also attend more and sleep better. A scatter plot shows
that two things move together, not *why*.

### Common chart mistakes

In [ ]:
# A chart without a title or labels: what are we looking at? Hours? Scores? Students?
plt.hist(students["sleep_hours"])
plt.show()

In [ ]:
# Mistake: plt.bar needs TWO things, the labels (x) and the heights
means = students.groupby("program")["final_score"].mean()
plt.bar(means)

## 6. How to choose a chart

| Your question | Chart | matplotlib |
|---------------|-------|------------|
| How are the values of **one** number spread out? | histogram | `plt.hist(column, bins=…)` |
| How does a number **compare across groups**? | bar chart | `plt.bar(labels, heights)` |
| Do **two numbers** move together? | scatter plot | `plt.scatter(x, y)` |
| How does something change **over time**? | line chart | `plt.plot(x, y)` |

Checklist for every chart: a title that says what it shows · axis labels **with units** · bars start at zero ·
one message per chart. Avoid pie charts with many slices and 3-D effects.

## 7. Project 06: Data Analysis Report

From today you have every tool for **Project 06**: load, clean, ask ≥ 5 questions, make ≥ 3 charts, and write up
findings and limitations. The brief is in `projects/06-data-analysis-report/README.md`; start from
`starter/report.ipynb`.

## Recap

1. Find problems: `isna().sum()`, `duplicated().sum()`, `value_counts()`, filters for impossible values.
2. Clean in a fixed, written-down order: `drop_duplicates()` → `.str.strip().str.lower()` → filter (watch out for
   missing values in conditions!) → `fillna(median)` (or `dropna()`, and say why).
3. Check your result against something you trust (shape, means).
4. Histogram = spread · bar = compare groups · scatter = two numbers · line = over time. Always title + axis labels.
5. A chart shows *that* things move together, not *why*.

**Next time:** machine learning with scikit-learn: using exactly this clean table to make predictions.